# Lab 15 — Real Open-Weight SFT on Free Colab

**Goal:** take a real Qwen3 open-weight checkpoint, establish a baseline, run a smoke test, then perform a small supervised fine-tuning experiment.

**Primary model:** `Qwen/Qwen3-0.6B`. If the runtime cannot provide enough memory, reduce sequence length/batch size or use the CPU path for dataset inspection and resource accounting.

**Important distinction:** SFT is the training objective. LoRA/QLoRA are parameter-update strategies and are taught separately.


In [ ]:
!pip -q install -U transformers datasets trl accelerate peft
import torch, transformers, trl
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('TRL:', trl.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))


## 1. Resource prediction
Before loading the model, predict its raw BF16 weight footprint from parameter count. Compare your estimate with measured GPU memory after loading.

In [ ]:
MODEL_ID='Qwen/Qwen3-0.6B'
PARAMETERS=0.6e9
print('Approx BF16 raw weights (GB):', PARAMETERS*2/1e9)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
)
print('Parameters:', sum(p.numel() for p in model.parameters()))
if torch.cuda.is_available():
    print('Allocated GiB:', round(torch.cuda.memory_allocated()/2**30, 2))


## 2. Baseline generation
Save the exact prompt and generation configuration. This output is the baseline that the trained model must beat or change in a measurable way.

In [ ]:
messages=[{'role':'user','content':'Explain gradient descent in two simple sentences.'}]
inputs=tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors='pt').to(model.device)
with torch.no_grad():
    out=model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))


## 3. Tiny inspectable dataset
This dataset is intentionally small. First print it. Then split it. In a real project, data quality and coverage are part of the experiment.

In [ ]:
from datasets import Dataset
examples=[
 {'text':'User: What is a GPU?\nAssistant: A GPU is a processor designed for highly parallel numerical computation.'},
 {'text':'User: Explain overfitting simply.\nAssistant: Overfitting happens when a model learns the training examples too closely and performs poorly on new examples.'},
 {'text':'User: Why use a validation set?\nAssistant: A validation set estimates performance on examples that were not used for fitting the model.'},
 {'text':'User: What is gradient descent?\nAssistant: Gradient descent updates model parameters in a direction that reduces the training loss.'},
]
data=Dataset.from_list(examples).train_test_split(test_size=0.25, seed=42)
print(data)
print(data['train'][0])


## 4. Smoke test
Do not begin with a long run. Run a tiny job first and verify that the loss changes and a checkpoint can be produced.

In [ ]:
from trl import SFTConfig, SFTTrainer
bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args=SFTConfig(
    output_dir='outputs/qwen3_0.6b_sft_smoke',
    max_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy='steps',
    save_steps=5,
    max_length=256,
    bf16=bf16,
    fp16=not bf16 and torch.cuda.is_available(),
    report_to=[],
)
trainer=SFTTrainer(model=model, args=args, train_dataset=data['train'], eval_dataset=data['test'], processing_class=tokenizer)
trainer.train()
trainer.save_model('outputs/qwen3_0.6b_sft_smoke/final')


## 5. Evaluation and failure analysis
Compare baseline and trained outputs using a held-out prompt set. Do not use training examples as your only evidence.

Record:
- task correctness;
- format validity;
- response length;
- unwanted behavior;
- retained capability;
- peak memory;
- wall time.

### Questions
1. Did loss decrease?
2. Did behavior change?
3. Did the target behavior improve?
4. Did any unrelated behavior regress?
5. What experiment would reduce the largest remaining uncertainty?


## Primary sources
- Qwen3-0.6B model card: https://huggingface.co/Qwen/Qwen3-0.6B
- TRL SFTTrainer: https://huggingface.co/docs/trl/sft_trainer
